# 03 — Model Training & Evaluation

In this notebook, we load the engineered features (Stage 3), and execute two full grid-searches to identify the best hyperparameters for **Support Vector Machine (RBF Kernel)** and **Random Forest** algorithms.

Due to our relatively small sample size ($n=73$), we avoid a simple Train/Test split in favor of **Repeated Stratified K-Fold Cross Validation** (5 splits, 3 repeats). This gives a more reliable estimate of how well the model generalizes to Fallers vs Non-Fallers.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.model_selection import cross_val_predict

from posturisk.preprocess import DEFAULT_PROCESSED_DIR
from posturisk.train import load_data, train_models

plt.rcParams.update({
    'figure.figsize': (8, 6),
    'font.size': 11,
    'axes.titleweight': 'bold',
})
%matplotlib inline

## 1. Execute Grid Searches

In [ ]:
features_path = DEFAULT_PROCESSED_DIR / "features.csv"
X, y = load_data(features_path)

print(f"Dataset: {X.shape[0]} subjects, {X.shape[1]} features.")
print(f"Classes: {y.value_counts().to_dict()}")

# This triggers the same pipeline as running python -m posturisk.train
results = train_models(X, y)
svm_gs = results['svm']
rf_gs = results['rf']
best_model_name = results['best_name']

print(f"\nWinner: **{best_model_name}**")

## 2. Evaluation Metrics Comparison
We can aggregate the metrics out of the CV dictionary and display them in a clean table.

In [ ]:
def extract_best_metrics(gs, model_name):
    idx = gs.best_index_
    return {
        "Model": model_name,
        "AUC-ROC": gs.cv_results_['mean_test_roc_auc'][idx],
        "Accuracy": gs.cv_results_['mean_test_accuracy'][idx],
        "Sensitivity": gs.cv_results_['mean_test_sensitivity'][idx],
        "Specificity": gs.cv_results_['mean_test_specificity'][idx],
        "F1 Score": gs.cv_results_['mean_test_f1'][idx],
    }

metrics = [
    extract_best_metrics(svm_gs, "SVM (RBF)"),
    extract_best_metrics(rf_gs, "Random Forest")
]

df_metrics = pd.DataFrame(metrics).set_index("Model")
df_metrics = df_metrics.round(3)

# Apply a gentle background gradient for readability
df_metrics.style.background_gradient(cmap='Blues', axis=0)

## 3. Confusion Matrices
Getting out-of-fold predictions to view aggregated Confusion Matrices across the CV.

In [ ]:
# To properly visualize this without leakage, we use cross_val_predict with the best identified hyperparameters.
# Note: cross_val_predict does not technically support repeated K-fold gracefully for confusion matrices,
# so we will use the best estimator and a single StratifiedKFold logic built into cross_val_predict under the hood.
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_oof = cross_val_predict(svm_gs.best_estimator_, X, y, cv=cv)
rf_oof = cross_val_predict(rf_gs.best_estimator_, X, y, cv=cv)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y, svm_oof, ax=axes[0], display_labels=["Non-Faller", "Faller"], cmap='Blues'
)
axes[0].set_title("SVM Confusion Matrix (OOF)")

ConfusionMatrixDisplay.from_predictions(
    y, rf_oof, ax=axes[1], display_labels=["Non-Faller", "Faller"], cmap='Oranges'
)
axes[1].set_title("Random Forest Confusion Matrix (OOF)")

plt.tight_layout()
plt.show()

## 4. Receiver Operating Characteristic (ROC) Curves

In [ ]:
# Get out-of-fold probabilistic predictions
svm_proba = cross_val_predict(svm_gs.best_estimator_, X, y, cv=cv, method='predict_proba')[:, 1]
rf_proba = cross_val_predict(rf_gs.best_estimator_, X, y, cv=cv, method='predict_proba')[:, 1]

fpr_svm, tpr_svm, _ = roc_curve(y, svm_proba)
roc_auc_svm = auc(fpr_svm, tpr_svm)

fpr_rf, tpr_rf, _ = roc_curve(y, rf_proba)
roc_auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(7, 6))
plt.plot(fpr_svm, tpr_svm, color='darkblue', lw=2, label=f'SVM (AUC = {roc_auc_svm:.3f})')
plt.plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f'Random Forest (AUC = {roc_auc_rf:.3f})')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (OOF Predictions)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()